# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [5]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 45

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [6]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [7]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [8]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and very exuberant Pirate. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [9]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [10]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [11]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Arrr! Ye be askin\' about exercises fer lower back pain, eh? Well, me hearty, I’ve got some treasure chests of info fer ye! Here be the finest exercises to help keep yer lower back shipshape:\n\n- **Cat-Cow Stretch**: Get down on yer hands and knees. Arch yer back up like a curious feline (the "cat"), then dip it down like a gentle cow. Do this 10-15 times to limber up yer spine!\n\n- **Bird Dog**: From yer hands and knees again, stretch out one arm and the opposite leg, hold fer 5 seconds, then switch sides. Do 10 repetitions per side to strengthen those core muscles!\n\n- **Pelvic Tilts**: Lie on yer back with knees bent. Tighten yer abs and tilt yer pelvis up slightly, flattenin\' yer back against the floor. Hold fer 10 seconds and repeat 8-12 times.\n\n- **Partial Crunches**: Lie on yer back, knees bent, arms crossed over yer chest. Raise yer shoulders just off the floor, hold briefly, then lower down. Do 8-12 reps to stoke yer core!\n\n- **Knee-to-Chest Stretch**: Lie on yer back

In [12]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

"Arrr, matey! Sleep be the mighty treasure chest of health, aye! When ye're catchin' those restful naps, yer body be doin' a fine repair job—tissues mend, hormones be workin' their magic, and yer mind be consolidatin' memories like a seasoned sailor chartin' new courses! Proper sleep—seven to nine hours of deep, quality slumber—keeps yer immune system strong as a fortress, stokes yer mental well-being, and keeps yer body in tip-top shape. Without it, ye risk bein' a tired boat adrift, more prone to illness, stress, and forgetfulness. So, hoist the anchor, set a regular bedtime, and give yerself the gift of good sleep—yarrr, for a healthy, happy life!"

In [13]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

"Arrr, matey! If ye be cursed with headaches and stress, there be some mighty fine natural remedies to get yer ship back on course! Here be the treasure map of relief:\n\n**For Headaches, hoist the sails of relief with:**\n- Drinking plenty o' water to stay hydrated — ho, hydrate or evaporate!\n- Applying a cold or warm compress to yer head or neck — a warm towel or an ice pack, yarrr!\n- Resting in a dark, quiet room — shiver me timbers, silence be golden!\n- Gently massaging yer temples and neck — give yerself a soothing rub!\n- Smelling peppermint or lavender essential oils — scents that calm the storm!\n- A small amount of caffeine — watch yer intake, it can help or hurt, yarrr!\n- Maintaining a regular sleep schedule — keep a steady course, matey!\n\n**And for stress, quick relief awaits:**\n- Deep breathing exercises — inhale for 4, hold, and exhale, like the calm seas!\n- Progressive muscle relaxation — tense and then release, from toes to head!\n- Grounding techniques — name 5 

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [ ]:
# Adding install
!pip install rank-bm25

In [16]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [17]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [18]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Arrr, matey! If yer walkin\' the plank into the treacherous waters of lower back pain, fear not! There be some mighty fine exercises to help ye sail smoothly again! \n\nFirst up, the Cat-Cow Stretch! Get down on yer hands and knees, and alternate between archin\' yer back up like a fearful feline and droppin\' it down like a lazy cow. Do 10-15 reps, and ye\'ll be shoutin\' "Yo-ho!" with relief!\n\nNext, the Bird Dog! From yer hands and knees, stretch out opposite arm and leg, holdin\' steady for 5 seconds. Switch sides and repeat 10 times on each side. Healthy backs, aye!\n\nAnd don\'t forget the Pelvic Tilts—lie on yer back, knees bent, and tighten yer abs to flatten yer back against the deck. Hold for 10 seconds, do 8-12 repetitions, and ye\'ll be feelin\' stronger in no time!\n\nSo, hoist the sails and try these exercises, matey, and ye\'ll be back to chartin\' a course of comfort soon enough! Yarrr!'

In [19]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Arr matey! Ye want to know how sleep affects overall health, eh? Well, shiver me timbers! Sleep be the loot of the kingdom of wellness! When ye get good, proper rest, yer body and mind be sailin’ smooth seas! It helps boost yer immune system, keeps yer brain sharp, and ensures yer gut be happy as a clam! A restful sleep can keep ye from ailin’, improve yer mood, and give ye the energy to conquer the day! Without enough sleep, ye might walk the plank into fatigue, confusion, and poor health, arrr! So, make sure ye keep a steady sleep schedule and create a cozy, dark, and quiet cabin for yer nightly voyage! Yo-ho-ho!'

In [20]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

"Arrr, matey! When it comes to battlin' stress and headaches naturally, there be a bounty o' remedies for ye! Here be some treasures from the health seas:\n\n1. **Herbal Teas:** Sip on chamomile or valerian root tea to soothe yer mind and calm yer noggin. Aye, the gentle waves of these herbs can ease tension!\n\n2. **Deep Breathing & Meditation:** Take in a deep breath, hold it tight, then exhale slow as the tide. Meditation and relaxation exercises can unravel the knots of stress.\n\n3. **Magnesium Supplements:** Consult yer healer first, but magnesium can help relax yer muscles and reduce headache pain.\n\n4. **Progressive Muscle Relaxation:** Tense up yer muscles, then release, lettin' the stress drift away like a ship on the calm sea.\n\n5. **Manage Triggers:** Stay well-hydrated, get enough sleep, and avoid foods that might stir up the storm—like processed meats or aged cheese.\n\nSo hoist the sails of calm and serenity, me hearty! These natural remedies can help ye navigate throu

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:

Example query: “What does section 5.2.1 of POLICY-SEC-042 say about rotating AWS access keys?”

BM25 is the perfect choice for the query because it's far better with exact wording, IDs, numbers, or exact phrases than embeddings are. Embeddings are much better at semantic similarity, synonyms, paraphrases, and 'fuzzy'/unclear/inaccurate queries.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [ ]:
!pip install -U langchain-classic

In [26]:
# Updating these import packages
from langchain_cohere import CohereRerank
from langchain_core.documents import Document
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [30]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [31]:
contextual_compression_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"Arrr, matey! If ye be seekin' relief fer yer lower back pain, hoist the sails and try out these fine exercises, yarrr! \n\nFirst, the Cat-Cow Stretch: Get down on all fours, then alternate archin’ yer back up like a curious feline (the cat), and saggin’ down like a contented cow. Do this 10 to 15 times, arrr! \n\nNext, the Bird Dog: From your hands and knees, stretch out yer opposite arm and leg like a daring albatross in flight, holdin’ steady for 5 seconds, then switch sides. Do this 10 times per side, yarrr! \n\nAnd don’t forget, ye can also try Pelvic Tilts: Lie on yer back, knees bent, then tighten yer abs and tilt yer pelvis up slightly to flatten yer back against the deck. Hold fer 10 seconds, repeat 8 to 12 times! \n\nThese gentle exercises can help keep yer lower back shipshape, matey!"

In [32]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

"Arrr, matey! Sleep be the treasure chest of good health, it be! When ye catch those splendid zzz's, yer body be workin' its magic—repairin tissues, processin' memories, and release'n hormones that help ye grow and hunt down yer appetite! A solid 7 to 9 hours of slumber every night keeps yer mind sharp, yer body strong, and yer spirit ready to sail the high seas! Without enough shut-eye, ye risk runnin' low on energy, battlin' insomnia, and sailin' into the stormy waters of poor health. So, hoist yer pillows, keep yer cabin cozy, and let sleep be yer loyal first mate on the voyage to wellness! Yo ho!"

In [33]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

"Arrr, ahoy matey! If ye be battlin' stress and headaches, there be some natural remedies in yer treasure chest! Drink plenty of water to keep yerself hydrated, aye! Apply a cold or warm compress to yer noggin or neck to soothe the pain. Restin' in a dark, quiet cabin be often just the trick. Gentle massagin' of yer temples and neck can ease the tension, and the fragrant oils o' peppermint or lavender may bring calm to yer stormy mind. Remember, keep yer sleep schedule regular as a ship's watch, and sometimes a wee bit of caffeine can help or hinder—be watchful! So, hoist the sails of relaxation and conquer yer stress and headaches, me hearty!"

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [35]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [36]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [37]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"Arrr, avast me hearty! If ye be seekin' exercises to battle that pesky lower back pain, hoist the sails and listen up! Here be some fine swashbucklin' moves to help ye keep that back shipshape:\n\n1. **Cat-Cow Stretch**: Start on yer hands and knees, then arc yer back up like a fierce feline (Cat) and then sag it down like a gentle cow (Cow). Do 10-15 repetitions to loosen the spine.\n\n2. **Bird Dog**: From the same position, extend opposite arm and leg while keepin' yer core tight as a treasure chest. Hold for 5 seconds, then switch sides. Do 10 reps on each side.\n\n3. **Partial Crunches**: Lie on yer back, knees bent, arms crossed over yer chest. Tighten yer stomach and lift yer shoulders off the deck, then lower down slowly. Aim for 8-12 repetitions.\n\n4. **Knee-to-Chest Stretch**: On yer back again, pull one knee toward yer chest, hold for 15-30 seconds, then switch. This helps stretch those back muscles.\n\n5. **Pelvic Tilts**: Lie on yer back with knees bent, flatten yer back

In [38]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Arrr, me hearty! Sleep be the mighty treasure chest for one’s overall health, aye! When ye get enough quality slumber—between 7 to 9 hours for grown-ups—ye be helpin’ yer body repair tissues, strengthen yer immune system, and keep yer mind sharp as a cutlass! Sleep also plays a crucial role in consolidatin’ memories, regulate hormones for growth and appetite, and keep yer spirits high! Without proper rest, ye risk bein’ plagued by fatigue, weakened defenses, and mental fog. So hoist the sails on good sleep habits, keep yer environment cozy and dark, and set a regular sleep schedule to keep yer health shipshape! Yarrr!'

In [39]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

"Arrr, matey! Ye be askin' about natural remedies for stress and headaches, eh? Well, shiver me timbers, there's plenty o' good ways to find relief without pillin' up! Here be some treasure troves of natural remedies:\n\nFor Headaches:\n- Drink water to stay hydrated, arrr!\n- Apply a cold or warm compress to yer head or neck to soothe the pain.\n- Rest in a dark, quiet room—perfect for catchin' some z's, aye!\n- Give yerself a gentle massage of yer temples and neck, like a true pirate healer.\n- Brew some peppermint or lavender essential oils—ah, the scents of the sea!\n- A small amount of caffeine might help, but beware, it can also hurt if ye overdo it.\n- Keep a regular sleep schedule—consistency is key to avoid headaches.\n\nFor Stress:\n- Take deep breaths—inhale for 4, hold for 4, exhale for 4, and repeat! It's like breathin' the calm of the ocean.\n- Try progressive muscle relaxation: tense and then release muscle groups from toes to head.\n- Use grounding techniques—name 5 thi

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:

Generating multiple reformulations of a user query improves recall because each version uses different words, structures, and levels of specificity which results in the retrieval of more and different documents than a single phrase retreival ever could. Some formulations bring to surface synonym-heavy matches, others surface ID-like terms or narrow subquestions. When the top results are merged across all these queries, many more relevant documents in the corpus are coverd, so the RAG step has richer and more complete context to answer from.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [41]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [47]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [48]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [49]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [50]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [51]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"Arrr, matey! If ye be battlin' with lower back pain, there be some fine exercises to help ye sail smoothly on the seas of health! Hoist yer sails and try these:\n\n- **Cat-Cow Stretch**: On yer hands and knees, arch yer back up like a fearful feline (cat), then sag down like a cow. Do 10-15 times to limber up yer spine!\n- **Bird Dog**: From the same position, stretch out opposite arm and leg, hold for 5 seconds, then switch. Do 10 repetitions per side to strengthen yer core!\n- **Partial Crunches**: Lie on yer back, knees bent, cross yer arms, and lift yer shoulders slightly off the deck to engage yer abs. Do 8-12 reps for a sturdy core!\n- **Knee-to-Chest Stretch**: Lie on yer back, pull one knee up to yer chest, hold for 15-30 seconds, then switch. This eases the tension in yer lower back!\n- **Pelvic Tilts**: Lie on yer back, knees bent, tighten yer abs and tilt pelvis slightly, hold for 10 seconds. Repeat 8-12 times to keep yer lower back shipshape!\n\nRemember, matey, always mov

In [52]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Arrr matey! Sleep be the treasure chest of good health, aye aye! When ye catch enough of the nightly rest — that be 7 to 9 hours for us adults — yer body be gettin’ a chance to repair tissues, settle yer mind, and release vital hormones that keep ye spry and hungry for adventure! Sleep be divided into cycles of about 90 minutes, swingin’ between light, deep, and REM sleep — each stage a crucial part of keepin’ yer muscles, brain, and heart in shipshape condition. \n\nSkip yer slumber, and ye risk feeling fatigued, suffer headaches, and even get dizzy, like a ship without a rudder! So, make yer sleep a priority, keep yer cabin chill and dark, and follow good sleep habits like a true captain. Enthusiasm for a good night’s rest is the secret to keepin’ yer health shipshape and ready for all of life’s voyages! Yarrr!'

In [53]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

"Arrr, me hearty! If ye be lookin' fer natural remedies to shoo away stress and headaches, here be the treasure ye seek:\n\n1. Hydration! Drink plenty o' water to keep yerself from dehydration, which be a common trigger fer headaches.\n2. Rest up! Find a dark, quiet room and give yerself a good nap or plenty o’ sleep — a proper slumber be legendary for head and stress relief.\n3. Gentle touch! A relaxing massage of yer temples and neck can turn the tide against headache pain.\n4. Aromatherapy! Peppermint and lavender essential oils be known to soothe headaches and calm yer mind.\n5. Warm or cold compress! Apply it to yer head or neck to lessen tension and ease pain.\n6. Mindfulness and deep breathing! Take slow, deep breaths, focus on the present, and let yer worries drift away like a ship on the horizon.\n7. Charming herbal teas! Chamomile or valerian root can help calm a restless mind and ease stress.\n\nRemember, matey, a combination of rest, hydration, and relaxation can be yer bes

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [54]:
from langchain_classic.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [55]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [65]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"Arrr! The hearty answer ye seek, matey! To battle that pesky lower back pain, ye can try these jolly good exercises:\n\n1. **Cat-Cow Stretch** — Get on yer hands and knees, then alternate archin' yer back up like a scurvy cat and let it sag down like a gentle cow. Do 10-15 repetitions to keep yer spine in shipshape!\n\n2. **Bird Dog** — From the same position, extend opposite arm and leg while keepin' yer core tight as a treasure chest. Hold for 5 seconds, then switch sides. Do this 10 times per side to strengthen yer back and core!\n\n3. **Pelvic Tilts** — Lie on yer back, knees bent, flatten yer back against the deck by tighten' yer abs and tiltin' yer pelvis up a smidge. Hold for 10 seconds, repeat 8-12 times to keep yer lower back sturdy as a ship's hull!\n\n4. **Partial Crunches** — Lie on yer back, knees bent, arms crossed over yer chest. Tighten yer stomach muscles and lift yer shoulders off the deck, then lower down. Do 8-12 reps to strengthen yer core!\n\n5. **Knee-to-Chest S

In [57]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

"Arrr matey, sleep be the treasure chest of good health, aye aye! When ye catch enough restful slumber—about 7-9 hours for grown pirates—ye be repairin yer tissues, buildin yer strength, and regulaatin' yer hormones, like growth and appetite. Sleep cycles through stages like light sleep, deep sleep, and dreams in REM sleep, each vital for brain power, memory, and feelin' refreshed. So, if ye skimp on sleep, ye risk take a hit to yer immune system, mental clarity, and overall vitality. It be the secret to keepin' yer body and mind sharp as a cutlass, arrr!"

In [58]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

"Arrr, matey! When ye be battlin' stress and headaches naturally, there be plenty o' treasure troves of remedies from Mother Nature herself! Here be some boons to ease yer troubles:\n\n💧 Stay Hydrated: Drink plenty of water to keep dehydration at bay, which be a common headache trigger.\n\n🌿 Aromatherapy: Peppermint or lavender essential oils can be applied to soothe yer senses and calm yer mind.\n\n🌙 Rest and Sleep: Maintain a regular sleep schedule, rest in a dark, quiet room, and consider relaxing bedtime routines like warm baths, reading, or gratitude journaling to keep stress at bay.\n\n🧘\u200d♂️ Relaxation Techniques: Practice deep breathing—inhale for 4, hold for 4, exhale for 4—progressive muscle relaxation, or mindfulness meditation to calm yer mind and body.\n\n🌞 Gentle Massage: Massaging yer temples and neck can help reduce muscle tension that be causin' headaches.\n\n🌬️ External Comforts: Apply cold or warm compresses to yer head or neck for quick relief.\n\n🌺 Herbal Allies

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [ ]:
!pip install langchain-experimental

In [61]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [62]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [63]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [66]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [67]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [68]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"Arrr, matey! To help ease that pesky lower back pain, ye can try these hearty exercises, aye aye! \n\nFirst, there's the **Cat-Cow Stretch**, where ye get on all fours, arrr, and alternate arcin' yer back up (the cat) and saggin' it down (the cow). Do 10-15 repetitions—ye'll feel quite the stretch! \n\nNext, the **Partial Crunches** be great! Lie on yer back, knees bent, cross yer arms, tighten yer belly muscles, and lift yer shoulders off the floor. Do 8-12 reps to strengthen yer core, which supports yer back! \n\nThen, ye can try the **Knee-to-Chest Stretch**—lie on yer back, pull one knee toward yer chest while keepin' the other foot flat on deck. Hold for 15-30 seconds, then switch sides. \n\nAnd don’t forget the **Pelvic Tilts**, where ye lie on yer back, knees bent, tighten yer abs and tilt yer pelvis up a bit, hold for 10 seconds, and repeat 8-12 times. This helps loosen up yer lower back, yarrr! \n\nRemember, start slow and build yer routine, and ye'll be back to sailin' smoot

In [69]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

"Arrr, matey! Sleep be the secret treasure chest for yer overall health, aye! When ye catch enough zzz’s—seven to nine hours per night—yer body gets the chance to repair tissues, strengthen yer immune defenses, and even sharpen yer mind for the adventures ahead! Proper sleep helps regulate hormones that control appetite and growth, keeps yer mood shipshape, and even boosts yer cognitive powers. Arrr, neglect yer sleep, and ye risk the dangers of fatigue, headaches, and a weakened immune system—harder to sail through life's storms, aye! So, make sleep a top priority, keep yer cabin dark and cool, and set a regular bedtime to keep yer health in shipshape condition! Yarrr!"

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

Percentile-based semantic chunking's algorithm measures embedding similarity between adjacent sentences and cuts where similarity drops past a selected percentile. In the case of FAQs with short, repetitive sentences, similarities span a tight band or narrow range, so the percentile threshold stops reflecting topic shifts resulting in almost no breakpoints and a huge grab-bag chunks of many FAQs instead, or splits at completely arbitrary and very much non-semantic points. Retrieval under these conditions will return big mixed chunks instead of the ideal one or two entries.

Improvements/adjustments

Add structural rules-
Treat each FAQ entry (question + answer) as a base unit and only merge neighboring entries when similarity is extremely high.

Tighten thresholds-
Set a lower percentile or an absolute similarity cutoff instead, tuned to this data. Also employ a more strict max chunk size, so chunk size can be kept near one or two or a few FAQs, instead of entire pages.



In [70]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

"Arrr, matey! When yer cravin' some natural remedies to chase away stress and headaches, here's what ye can do, aye aye!\n\n1. Drink plenty of water, so ye stay hydrated— dehydration be a common trigger fer headaches!  \n2. Rest in a dark, quiet room and give yer noggin a break—ye might try a cold or warm compress on yer head or neck.  \n3. Use calming essential oils like peppermint or lavender—smellin' 'em can soothe yer mind and ease headaches.  \n4. Practice deep breathing—inhale fer 4 counts, hold, then exhale, repeat—this be a quick way to calm yer nerves!  \n5. Engage in gentle stretching or muscle relaxation—progressive muscle relaxation can help ease tension throughout yer body.  \n6. Try mindfulness or meditation—focusing on yer breath and bein' present can reduce the stress that leads to headaches.  \n7. And don’t forget to take short walks in nature—fresh air and a change of scenery can do wonders!  \n\nSo, hoist yer sails and try these natural remedies, matey! They be as tr

---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [ ]:
### YOUR CODE HERE